## 1. Config
Fill in your API key ID and path to your RSA private key `.pem` file.

In [ ]:
BASE_URL         = "https://api.elections.kalshi.com/trade-api/v2"
API_KEY_ID       = "your-api-key-id"         # KALSHI-ACCESS-KEY
PRIVATE_KEY_PATH = "kalshi_private_key.pem"  # path to your RSA private key

## 2. Auth Helpers

In [ ]:
import base64
import time
import requests
from pathlib import Path
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding


def load_private_key(path: str):
    pem = Path(path).read_bytes()
    return serialization.load_pem_private_key(pem, password=None)


def sign_request(private_key, method: str, path: str, timestamp_ms: int) -> str:
    message = f"{timestamp_ms}{method}{path}".encode()
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH,
        ),
        hashes.SHA256(),
    )
    return base64.b64encode(signature).decode()


def auth_headers(private_key, method: str, path: str) -> dict:
    ts = int(time.time() * 1000)
    return {
        "KALSHI-ACCESS-KEY":       API_KEY_ID,
        "KALSHI-ACCESS-TIMESTAMP": str(ts),
        "KALSHI-ACCESS-SIGNATURE": sign_request(private_key, method, path, ts),
        "Content-Type":            "application/json",
    }


private_key = load_private_key(PRIVATE_KEY_PATH)
print("✅ Private key loaded")

## 3. Fetch Candlesticks

In [ ]:
def get_batch_candlesticks(
    private_key,
    market_tickers: list,
    start_ts: int,
    end_ts: int,
    period_interval: int = 1,           # 1 = 1 min | 60 = 1 hr | 1440 = 1 day
    include_latest_before_start: bool = False,
) -> list:
    """
    Fetch candlesticks for up to 100 markets at once.
    Returns a list of { market_ticker, candlesticks: [...] }
    """
    path = "/trade-api/v2/markets/candlesticks"
    params = {
        "market_tickers":              ",".join(market_tickers),
        "start_ts":                    start_ts,
        "end_ts":                      end_ts,
        "period_interval":             period_interval,
        "include_latest_before_start": str(include_latest_before_start).lower(),
    }
    headers  = auth_headers(private_key, "GET", path)
    response = requests.get(BASE_URL + "/markets/candlesticks", headers=headers, params=params)
    response.raise_for_status()
    return response.json().get("markets", [])

In [ ]:
# ── Set your parameters here ──────────────────────────────────────────────────

MARKET_TICKERS  = ["INXD-24JAN01", "BTCZ-24JAN01"]  # up to 100 tickers
PERIOD_INTERVAL = 1    # 1 = 1 min | 60 = 1 hr | 1440 = 1 day

now      = int(time.time())
START_TS = now - 7 * 24 * 60 * 60   # 7 days ago
END_TS   = now

# ─────────────────────────────────────────────────────────────────────────────

raw = get_batch_candlesticks(
    private_key=private_key,
    market_tickers=MARKET_TICKERS,
    start_ts=START_TS,
    end_ts=END_TS,
    period_interval=PERIOD_INTERVAL,
)

print(f"Fetched {len(raw)} market(s)")
for m in raw:
    print(f"  {m['market_ticker']}: {len(m['candlesticks'])} candles")

## 4. Parse into a DataFrame

In [ ]:
import pandas as pd


def parse_candlesticks(raw: list) -> pd.DataFrame:
    rows = []
    for market in raw:
        ticker = market["market_ticker"]
        for c in market["candlesticks"]:
            p = c.get("price", {})
            rows.append({
                "ticker":        ticker,
                "timestamp":     pd.to_datetime(c["end_period_ts"], unit="s", utc=True),
                "open":          p.get("open"),          # cents
                "high":          p.get("high"),
                "low":           p.get("low"),
                "close":         p.get("close"),
                "open_usd":      p.get("open",  0) / 100,  # dollars
                "high_usd":      p.get("high",  0) / 100,
                "low_usd":       p.get("low",   0) / 100,
                "close_usd":     p.get("close", 0) / 100,
                "volume":        c.get("volume", 0),
                "open_interest": c.get("open_interest", 0),
            })
    df = pd.DataFrame(rows)
    df = df.sort_values(["ticker", "timestamp"]).reset_index(drop=True)
    return df


df = parse_candlesticks(raw)
df.head(10)

## 5. Quick Summary

In [ ]:
df.groupby("ticker").agg(
    candles=("timestamp", "count"),
    from_=("timestamp", "min"),
    to=("timestamp", "max"),
    avg_close_usd=("close_usd", "mean"),
    total_volume=("volume", "sum"),
).round(4)

## 6. Plot Close Price

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))

for ticker, group in df.groupby("ticker"):
    ax.plot(group["timestamp"], group["close_usd"], label=ticker)

ax.set_title("Kalshi Market Close Price (USD)")
ax.set_xlabel("Time")
ax.set_ylabel("Price ($)")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Export to CSV (optional)

In [ ]:
df.to_csv("kalshi_candlesticks.csv", index=False)
print(f"Saved {len(df)} rows to kalshi_candlesticks.csv")